# EDA 3 — Typology & Validation
**Phases E + F of the EDA pipeline**

Constructs the flow regime typology from Cascade_Dominance × Cross_Decile_Share,
profiles each type, and validates against IMD_Pctile_Change.

### Depends on
`msoa_cascade_features_enriched_20260616.csv` produced by `eda_1_metric_landscape_20260616.ipynb`.

---

## 12. Setup & Load Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
from pyprojroot import here

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'})

ROOT = here()
DATA_DIR   = ROOT / 'outputs'
OUTPUT_DIR = ROOT / 'outputs/eda_figs'  
GEO_PATH   = ROOT / 'data/london_msoa_2011.geojson'  
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


from map_utils import load_london_msoa, plot_london_choropleth, plot_london_categorical

df = pd.read_csv(DATA_DIR / 'msoa_cascade_features_enriched_20260616.csv')
print(f'Loaded: {df.shape[0]} MSOAs, {df.shape[1]} columns')

Loaded: 983 MSOAs, 61 columns


---
## Phase E — Classifying MSOA Flow Regimes

### Typology Axes
- **Cascade_Dominance** (x-axis): which direction of deprivation-crossing flow dominates?
  - \> 0.50 → cascade-led (net downward pressure on existing residents)
  - < 0.50 → counter-led (net upward mobility signal)
  - ≈ 0.50 → symmetric (roughly balanced)
- **Cross_Decile_Share** (y-axis): how much of total migration crosses decile lines?
  - High → most migration involves socioeconomic restructuring
  - Low → most migration stays within the same deprivation tier (lateral)

---
## 13. Typology Scatter — Exploring the 2D Space

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    dom = f'Cascade_Dominance_{yr}'
    cds = f'Cross_Decile_Share_{yr}'
    
    scatter = ax.scatter(
        df[dom], df[cds],
        c=df['Wealth_Decile'], cmap='YlOrBr_r',
        s=12, alpha=0.5, edgecolor='none', vmin=1, vmax=10
    )
    
    ax.axvline(0.5, color='black', ls=':', lw=1, alpha=0.5, label='Dominance midline')
    ax.set_xlabel('Cascade Dominance')
    ax.set_ylabel('Cross-Decile Share')
    ax.set_title(f'20{yr}', fontsize=12)
    ax.set_xlim(0.2, 0.75)
    ax.set_ylim(0.4, 1.0)

plt.colorbar(scatter, ax=axes, label='Wealth Decile', shrink=0.7, pad=0.02)
fig.suptitle('Typology Space: Cascade Dominance × Cross-Decile Share',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_12_typology_scatter_exploration.png')
plt.show()

# Print descriptive stats for the two axes
for yr in ['21']:
    print(f'\n=== 20{yr} ===')
    print(df[[f'Cascade_Dominance_{yr}', f'Cross_Decile_Share_{yr}']].describe().round(4).to_string())

##### Threshold exploration

Before committing to typology cutoffs, let's examine natural break points.
The scatter above should reveal whether there are natural clusters or whether
the data is continuous (requiring percentile-based thresholds).

---
## 14. Typology Construction

Based on the scatter exploration, we define thresholds. The initial approach
uses the natural midline (0.50) for Cascade_Dominance and a percentile-based
threshold for Cross_Decile_Share.

**Thresholds should be reviewed after examining the scatter above.**

In [ ]:
def assign_typology(row, yr='21',
                    dom_upper=0.52, dom_lower=0.48,
                    cds_threshold=None):
    """
    Classify MSOA into flow regime typology.
    
    Parameters
    ----------
    dom_upper, dom_lower : float
        Cascade_Dominance thresholds. 
        > dom_upper → cascade-led; < dom_lower → counter-led; between → symmetric
    cds_threshold : float
        Cross_Decile_Share below this → lateral-dominated.
        Default: 25th percentile of CDS distribution.
    """
    dom = row[f'Cascade_Dominance_{yr}']
    cds = row[f'Cross_Decile_Share_{yr}']
    
    if cds < cds_threshold:
        return 'Lateral'
    elif dom > dom_upper:
        return 'Cascade-led'
    elif dom < dom_lower:
        return 'Counter-led'
    else:
        return 'Symmetric'

# Compute threshold (25th percentile of Cross_Decile_Share)
cds_p25 = df['Cross_Decile_Share_21'].quantile(0.25)
print(f'Cross_Decile_Share 25th percentile (2021): {cds_p25:.4f}')

# Apply typology
df['Typology_21'] = df.apply(
    assign_typology, axis=1, yr='21', cds_threshold=cds_p25
)

# Also for 2011
cds_p25_11 = df['Cross_Decile_Share_11'].quantile(0.25)
df['Typology_11'] = df.apply(
    assign_typology, axis=1, yr='11', cds_threshold=cds_p25_11
)

print(f'\n=== Typology Distribution (2021) ===')
print(df['Typology_21'].value_counts().to_string())
print(f'\n=== Typology Distribution (2011) ===')
print(df['Typology_11'].value_counts().to_string())

In [ ]:
# ── Typology scatter — coloured by type ───────────────────────
TYPOLOGY_COLORS = {
    'Cascade-led':  '#4393c3',   # cool blue — downward pressure
    'Counter-led':  '#d6604d',   # warm coral — upward mobility
    'Symmetric':    '#8073ac',   # muted purple — balanced
    'Lateral':      '#b8b8b8',   # grey — insulated
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    dom = f'Cascade_Dominance_{yr}'
    cds = f'Cross_Decile_Share_{yr}'
    typ = f'Typology_{yr}'
    
    for tname, tcolor in TYPOLOGY_COLORS.items():
        mask = df[typ] == tname
        ax.scatter(df.loc[mask, dom], df.loc[mask, cds],
                   c=tcolor, s=15, alpha=0.6, edgecolor='none',
                   label=f'{tname} ({mask.sum()})')
    
    ax.axvline(0.5, color='black', ls=':', lw=1, alpha=0.4)
    ax.set_xlabel('Cascade Dominance')
    ax.set_ylabel('Cross-Decile Share')
    ax.set_title(f'20{yr}')
    ax.legend(fontsize=8, loc='lower left')

fig.suptitle('MSOA Flow Regime Typology', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_13_typology_scatter_classified.png')
plt.show()

---
## 15. Typology — Diagnostic Map

In [ ]:
gdf = load_london_msoa(GEO_PATH, df)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    plot_london_categorical(
        gdf, column=f'Typology_{yr}',
        title=f'MSOA Flow Regime Typology (20{yr})',
        color_dict=TYPOLOGY_COLORS,
        ax=ax
    )

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_14_typology_map.png')
plt.show()

---
## 16. Typology Profiles

For each type: mean metrics, decile composition, borough composition.

In [ ]:
# ── Mean metrics per typology ──────────────────────────────────
profile_cols = [
    'CFI_Churn_21', 'Counter_Churn_21', 'Net_Cascade_21', 'Net_Counter_21',
    'CFI_Rate_21', 'Counter_Rate_21',
    'Pct_Inflow_Wealthier_21', 'Pct_Outflow_Wealthier_21',
    'Cascade_Dominance_21', 'Cross_Decile_Share_21',
    'Total_Migration_21', 'IMD_Pctile_Change'
]

print('=== Mean Metric Values by Typology (2021) ===')
profile = df.groupby('Typology_21')[profile_cols].mean().round(2)
print(profile.T.to_string())

In [ ]:
# ── Decile composition per typology ────────────────────────────
ct_decile = pd.crosstab(df['Typology_21'], df['Wealth_Decile'], normalize='index')
ct_decile = (ct_decile * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 5))
ct_decile.plot(kind='bar', stacked=True, cmap='YlOrBr_r', ax=ax, edgecolor='white', lw=0.3)
ax.set_xlabel('Typology')
ax.set_ylabel('% of MSOAs in typology')
ax.set_title('Wealth Decile Composition by Typology (2021)')
ax.legend(title='Decile', bbox_to_anchor=(1.02, 1), fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_15_typology_decile_composition.png')
plt.show()

In [ ]:
# ── Top boroughs per typology ──────────────────────────────────
print('=== Top 5 Boroughs per Typology (2021) ===')
for typ in ['Cascade-led', 'Counter-led', 'Symmetric', 'Lateral']:
    subset = df[df['Typology_21'] == typ]
    top = subset['ladnm'].value_counts().head(5)
    print(f'\n{typ} ({len(subset)} MSOAs):')
    for borough, count in top.items():
        print(f'  {borough}: {count}')

---
## Phase F — Does the Typology Predict Deprivation Change?

IMD_Pctile_Change is the external validation — constructed independently from
the cascade metrics. Positive = moved up in national ranking = less deprived.

---
## 17. Correlation with IMD_Pctile_Change

In [ ]:
# ── Scatterplots: key metrics vs IMD_Pctile_Change ────────────
validation_metrics = [
    'Net_Cascade_21', 'Net_Counter_21',
    'CFI_Churn_21', 'Counter_Churn_21',
    'Cascade_Dominance_21', 'Cross_Decile_Share_21'
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, col in enumerate(validation_metrics):
    ax = axes.flatten()[i]
    ax.scatter(df[col], df['IMD_Pctile_Change'], s=8, alpha=0.3, edgecolor='none')
    
    # Spearman correlation
    rho, pval = stats.spearmanr(df[col], df['IMD_Pctile_Change'])
    ax.set_title(f'{col.replace("_21", "")}\nρ = {rho:.3f} (p = {pval:.1e})',
                 fontsize=10)
    ax.set_xlabel(col)
    ax.set_ylabel('IMD Pctile Change')
    
    # Regression line
    z = np.polyfit(df[col], df['IMD_Pctile_Change'], 1)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), color='#b2182b', lw=1.5, ls='--')

fig.suptitle('Cascade & Counter-Cascade Metrics vs IMD Percentile Change',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_16_validation_scatters.png')
plt.show()

In [ ]:
# ── Partial correlations (controlling for Wealth_Decile) ──────
from scipy.stats import spearmanr

def partial_spearman(x, y, z):
    """Partial Spearman correlation of x and y, controlling for z."""
    # Rank-based: correlate residuals from regressing ranks on z
    from numpy.polynomial.polynomial import polyfit, polyval
    rx, ry, rz = stats.rankdata(x), stats.rankdata(y), stats.rankdata(z)
    # Residualize x and y on z
    cx = np.polyfit(rz, rx, 1)
    cy = np.polyfit(rz, ry, 1)
    resid_x = rx - np.polyval(cx, rz)
    resid_y = ry - np.polyval(cy, rz)
    return stats.spearmanr(resid_x, resid_y)

print('Validation: Spearman ρ with IMD_Pctile_Change')
print('=' * 70)
print(f'{"Metric":>28s} {"Raw ρ":>10s} {"Partial ρ":>10s} {"(ctrl W.D.)":>12s}')
print('─' * 70)

for col in validation_metrics:
    rho_raw, p_raw = stats.spearmanr(df[col], df['IMD_Pctile_Change'])
    rho_part, p_part = partial_spearman(df[col], df['IMD_Pctile_Change'], df['Wealth_Decile'])
    print(f'{col.replace("_21",""):>28s} {rho_raw:>+10.3f} {rho_part:>+10.3f} {"(p=" + f"{p_part:.1e}" + ")":>12s}')

---
## 18. Typology vs IMD Percentile Change

The ultimate validation: does the typology separate neighbourhoods with
different deprivation trajectories?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

order = ['Cascade-led', 'Symmetric', 'Counter-led', 'Lateral']
colors = [TYPOLOGY_COLORS[t] for t in order]

bp = df.boxplot(column='IMD_Pctile_Change', by='Typology_21',
                positions=range(len(order)), ax=ax, patch_artist=True,
                return_type='dict', widths=0.6)

# Style the boxes
for i, (box, median) in enumerate(zip(bp['IMD_Pctile_Change']['boxes'],
                                       bp['IMD_Pctile_Change']['medians'])):
    box.set_facecolor(colors[i])
    box.set_alpha(0.6)
    median.set_color('black')
    median.set_linewidth(2)

ax.set_xticklabels(order, fontsize=10)
ax.set_xlabel('Flow Regime Typology')
ax.set_ylabel('IMD Percentile Change (positive = less deprived)')
ax.set_title('')
fig.suptitle('IMD Percentile Change by Flow Regime Typology (2021)', fontsize=13)

# Kruskal-Wallis test
groups = [df.loc[df['Typology_21'] == t, 'IMD_Pctile_Change'].values for t in order]
kw_stat, kw_p = stats.kruskal(*groups)
ax.annotate(f'Kruskal-Wallis H = {kw_stat:.1f}, p = {kw_p:.2e}',
            xy=(0.98, 0.02), xycoords='axes fraction', ha='right', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_17_typology_vs_imd_boxplot.png')
plt.show()

---
## 19. Export Typology Labels

In [ ]:
# Export typology labels for downstream use (File 4 + 5)
export_cols = ['msoa11cd', 'ladnm', 'Wealth_Decile',
               'Typology_11', 'Typology_21',
               'Cascade_Dominance_21', 'Cross_Decile_Share_21',
               'IMD_Pctile_Change', 'Sign_Concordance_21']

export_df = df[export_cols].copy()
export_path = DATA_DIR / 'msoa_typology_labels.csv'
export_df.to_csv(export_path, index=False)

print(f'Typology labels exported: {export_path}')
print(f'Shape: {export_df.shape}')
print(f'\nTypology (2021):')
print(export_df['Typology_21'].value_counts().to_string())

---
## 20. Summary

### Phase E findings
- The typology space (Cascade_Dominance × Cross_Decile_Share) produces
  four interpretable flow regime types.
- Threshold sensitivity: the classification depends on the dominance bandwidth
  (±0.02 from 0.50) and the CDS percentile cutoff (25th). These should be
  tested for robustness.

### Phase F findings
- IMD_Pctile_Change validation: [to be filled after running]
- Partial correlations controlling for Wealth_Decile reveal which metrics
  carry information beyond hierarchy position.
- Typology boxplot: if the Kruskal-Wallis test is significant, the typology
  adds explanatory value for deprivation trajectories.

### Output
`msoa_typology_labels.csv` — consumed by File 4 (spatial synthesis) and
File 5 (case studies).

---
*Next: `eda_4_spatial_synthesis.ipynb` (Phase G)*